# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primary task: ranking / scoring in a hybrid ML pipeline.**

The system must answer: **which content pages should get attention first, why, and what action should be considered?** That makes ranking/scoring the final task.

The ranking is built in stages:

1. **Signal analysis** creates a leakage-safe feature set from observed search, traffic, engagement, freshness, and content data.
2. **Clustering** groups similar pages into performance archetypes and gives each page a fair peer group.
3. **Peer-relative analysis** measures how unusual a page is compared with similar pages, for example whether its CTR or engagement is weak for its archetype.
4. **Classification / prediction** estimates a future observed performance state using only information available before the decision point.
5. **Impact estimation** combines predicted risk, expected size of the change, and page exposure/value.
6. **Ranking / scoring** orders pages by expected adverse impact.
7. **Action generation** uses the archetype, predicted direction, and strongest abnormal signals to produce reason codes and a suggested action.

In short:

**observed signals → archetype → relative abnormalities → future prediction → expected impact → ranked priority → suggested action**

The suggested action is an evidence-based **intervention hypothesis**, not a proven causal effect. We can test prediction quality retrospectively and later test whether the recommended action causes improvement with a controlled or otherwise valid causal study.

In [1]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

methodology = [
    "signal analysis",
    "clustering / peer context",
    "archetype-relative deviations",
    "future-state prediction",
    "impact estimation",
    "ranking / scoring",
    "action hypothesis",
]
print("Methodology:", " -> ".join(methodology))


Rows: 30,000
Columns: 44
Unique content items: 30,000
Pseudonymized clients: 32
Methodology: signal analysis -> clustering / peer context -> archetype-relative deviations -> future-state prediction -> impact estimation -> ranking / scoring -> action hypothesis


## 2. Target or proxy

Different stages use different learning objects.

### Clustering

Clustering has **no target column**. It creates performance archetypes and peer baselines from the training data.

### Supervised prediction

The main supervised target should be an **observed future outcome**, not a manually created priority score or action label.

The intended structure is:

**past feature window → decision point → non-overlapping future outcome window**

The later warehouse target will be a future decline outcome such as `future_decline_30d`, built from observed search performance after the decision point. A continuous future-change value will also keep the size of the movement, so a small decline and a severe decline are not treated as equal.

The exact decline threshold, persistence rule, and minimum-volume floor are **not fixed in this notebook**. They will be defined in the data-contract stage and tested for sensitivity before model training.

### Starter-data proxy

The 30,000-row starter CSV is only a trailing-90-day snapshot, so it cannot provide a true future target. For Assignment 3, the transparent proxy is:

[
\text{decline\_proxy}=1
\quad \text{if} \quad
\texttt{trend\_direction = "down"}
]

This is a **defined current-window proxy**, not a future ground-truth outcome and not proof that a page needs intervention.

Because `trend_direction` comes from `trend_pct`, neither field can be used as a predictive feature when this proxy is the target.

### Ranking and action

The final ranking will not learn a hand-written priority label. Conceptually:

[
\text{priority}
\propto
P(\text{future decline})
\times
E(\text{decline magnitude})
\times
\text{measured exposure/value}
]

The exact scaling will be fixed later and validated rather than assigned arbitrary weights.

The action stage maps the page's archetype, predicted future state, and strongest peer-relative abnormalities to a **recommended intervention hypothesis** that can later be tested.

In [2]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_change_magnitude",
    ],
    "role": [
        "future-state classification target",
        "future-movement magnitude outcome",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


Starter proxy: decline_proxy = 1 when trend_direction == 'down'.
This is a current-window proxy, not a future causal or intervention label.

decline_proxy
0    13738
1    16262
Name: count, dtype: int64
Proxy positive rate: 54.2%



,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,decline_proxy
0,content_304f48230142,987,578,-41.4,down,1
1,content_a1fb4e703a9e,5915,2501,-57.7,down,1
2,content_9aa793d4d895,6089,2382,-60.9,down,1
3,content_331d6c4de07b,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,6452,4211,-34.7,down,1
5,content_d4084a4bc775,1009,617,-38.9,down,1
6,content_9a34b442b552,13,1,-92.3,down,1
7,content_a63219c6e95a,632,636,0.6,stable,0
8,content_5e6c160719bc,13828,5696,-58.8,down,1
9,content_c27558df2b0c,356,252,-29.2,down,1


Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']


,field,role,available_in_starter_snapshot
0,future_decline_30d,future-state classification target,false
1,future_change_magnitude,future-movement magnitude outcome,false


## 3. Success metric

**Primary end-to-end metric: Precision@K for the ranked human-review queue.**

At a fixed review capacity K, Precision@K asks:

> Of the K pages placed at the top of the review queue, what fraction later show the observed adverse outcome the system was trying to catch?

For the later warehouse evaluation, relevance must come from a **held-out future outcome after the decision point**, not from a hand-written priority label. The starter `decline_proxy` is useful only for checking the framing and metric plumbing; it is not evidence that the final model predicts future decline.

Precision@K matches the final decision because reviewer capacity is limited and the quality of the **top of the queue** matters more than average performance across all pages.

### Success metrics for the pipeline sub-tasks

| Sub-task | Primary success metric | What success means |
|---|---|---|
| Signal analysis | **Out-of-sample effect size / association stability** | Candidate signals show a measurable relationship with the later observed outcome and the direction/magnitude is reasonably stable across validation slices, rather than appearing only in one sample. |
| Clustering / archetypes | **Silhouette score**, plus a human sense-check | Clusters are more internally similar than externally similar and produce interpretable performance archetypes. A high silhouette score alone is not enough if the groups are not useful or stable. |
| Peer-relative abnormality | **Lift in future-outcome rate across deviation-score quantiles** | Pages with more abnormal peer-relative signals should show a higher later adverse-outcome rate than less abnormal pages. The relationship should be monotonic or at least directionally consistent. |
| Future-state classification | **PR-AUC**, compared with the outcome base rate | The classifier should rank true future adverse outcomes above non-events better than a no-skill prevalence baseline. Precision and recall at the operating threshold are secondary checks. |
| Future-change magnitude | **MAE** on the held-out future change value | Predicted decline magnitude should have lower absolute error than a simple constant or baseline prediction on the same held-out rows. |
| Impact estimation | **Spearman rank correlation** with realized future adverse impact | Higher estimated impact should correspond to worse realized future impact without requiring the estimate to be perfectly calibrated in absolute units. |
| Final ranking / scoring | **Precision@K** versus the transparent rule baseline | The learned queue must place more true future adverse outcomes in the top K than the rule baseline on the same rows and target. This is the primary project metric. |
| Action / reason-code generation | **Reviewer acceptance / traceability rate** | Suggested actions and reason codes should be understandable and supported by the measured signals. This evaluates decision support only; whether an action actually causes improvement requires a later controlled or otherwise valid causal study. |

### What number means "good"?

For the **final ranking**, the learned method must beat the transparent rule baseline on the **same rows, same future target, and same K**:

$$\Delta P@K = P@K_{\text{model}} - P@K_{\text{baseline}} > 0$$

Equivalently, `Lift@K = model Precision@K / baseline Precision@K` must be greater than 1.0.

For the other sub-tasks, success is also defined **relative to an appropriate baseline or validation expectation**, not by invented universal thresholds. For example, classification PR-AUC should exceed the positive-outcome prevalence; magnitude MAE should beat a constant baseline; clustering should outperform obviously weak or unstable partitions while remaining interpretable.

The operational value of K should come from the real human-review budget. Until that capacity is known, a fixed reporting depth such as `K = 50` can be used for reproducible comparison without pretending that 50 is the business optimum.

If a more complex stage fails to improve the relevant validation metric or does not add useful, interpretable information, that stage should be simplified or removed.

In [ ]:
# Metric contract for the pipeline and final ranked review queue.
import numpy as np
import pandas as pd

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if len(y_true) != len(scores):
        raise ValueError("y_true and scores must have the same length")
    if not 1 <= k <= len(y_true):
        raise ValueError("k must be between 1 and the number of rows")
    order = np.argsort(-scores, kind="stable")
    return float(y_true[order[:k]].mean())

metric_contract = pd.DataFrame({
    "sub_task": [
        "signal analysis",
        "clustering / archetypes",
        "peer-relative abnormality",
        "future-state classification",
        "future-change magnitude",
        "impact estimation",
        "final ranking / scoring",
        "action / reason codes",
    ],
    "primary_metric": [
        "out-of-sample effect-size / association stability",
        "silhouette score + interpretability check",
        "future-outcome lift across deviation quantiles",
        "PR-AUC vs outcome prevalence",
        "MAE vs constant baseline",
        "Spearman rank correlation with realized impact",
        "Precision@K vs rule baseline",
        "reviewer acceptance / traceability rate",
    ],
})

reporting_k = min(50, len(proxy_frame))
starter_proxy_rate = float(proxy_frame["decline_proxy"].mean())

display(metric_contract)
print(f"Primary end-to-end metric: Precision@K")
print(f"Current reporting depth: K = {reporting_k}")
print(f"Starter proxy prevalence: {starter_proxy_rate:.1%}")
print("Final-ranking success criterion:")
print("  model Precision@K - rule-baseline Precision@K > 0")
print("  equivalently, Lift@K > 1.0")
print("All comparisons must use the same held-out future target and evaluation rows.")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.